In [3]:
from sentence_transformers import       

model = SentenceTransformer('all-MiniLM-L6-v2')

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

In [4]:
q1 = 'Can I still join the course after the start date?'
v1 = model.encode(q1)

In [3]:
d  = "You don't need to register. You're accepted. You can also just start learning and submitting homework without registering."
dv = model.encode(d)

In [4]:
v1.dot(dv)

np.float32(0.32332397)

In [5]:
q2 = 'How to install Docker on Windows?'
v2 = model.encode(q2)

In [6]:
v2.dot(dv)

np.float32(0.019730574)

In [6]:
from ingest import load_faq_data

documents = load_faq_data()

In [7]:
texts = []

for doc in documents:
    text = doc['question'] + ' ' + doc['answer']
    texts.append(text)

In [8]:
from tqdm.auto import tqdm

In [9]:
batch_size = 50
vectors = []

for i in tqdm(range(0, len(texts), batch_size)):
    batch = texts[i:i + batch_size]
    batch_vectors = model.encode(batch)
    vectors.extend(batch_vectors)

len(vectors)

  0%|          | 0/25 [00:00<?, ?it/s]

1208

In [10]:
import numpy as np
X = np.array(vectors)

In [16]:
query = 'Can I still join the course after the start date?'
v_query = model.encode(query)

In [17]:
scores = X.dot(v_query)

In [18]:
idx = np.argmax(scores)
idx, scores[idx]

(np.int64(553), np.float32(0.762941))

In [2]:
documents[idx]

NameError: name 'documents' is not defined

In [11]:
from minsearch import VectorSearch

vindex = VectorSearch(keyword_fields=['course'])
vindex.fit(X, documents)

In [13]:
query = 'I just discovered the course. Can I still join it?'
query_vector = model.encode(query)

results = vindex.search(query_vector, num_results=5)
results

[{'id': '74eb249bbf',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'I just discovered the course. Can I still join?',
  'answer': 'Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.'},
 {'id': '41aabbd7c5',
  'course': 'machine-learning-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'The course has already started. Can I still join it?',
  'answer': 'Yes, you can. Even though you missed the start date, you can register for the course. You won’t be able to submit some of the homeworks, but you can still take part in the course.\n\nIn order to get a certificate, you need to submit 2 out of 3 course projects and review 3 peers by the deadline. It means that if you join the course at the end of November and manage to work on two projects, you will still be eligible for a certificate.'},
 {'id': '2d8b16c2a0',
  'course': 'mlops-zoomcamp',
  'section':